In [1]:
# Transformers example -> Reverseing sequence of numbers


In [34]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

import haiku as hk
import optax

from probjax.nn.transformers import Transformer, LearnedPosEmbed, PosEmbed

In [35]:
VOCAB_SIZE = 10

In [36]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)
    # Just reversing
    # sequences_reversed = jnp.array(jnp.flip(sequences, axis=-2), dtype=jnp.int32)
    # Sorting
    sequences_reversed = jnp.sort(sequences, axis=-2)

    
    return sequences, sequences_reversed


inputs, labels = generate_data(jrandom.PRNGKey(0), 100, 20)

In [37]:
key = jrandom.PRNGKey(0)

In [38]:
inputs.shape

(100, 20, 1)

In [26]:


@hk.transform
def f(x):
    # Embed
    x = jnp.squeeze(hk.Embed(VOCAB_SIZE, 20)(x), -2)
    # x = jnp.squeeze(jax.nn.one_hot(x, VOCAB_SIZE))
    x = PosEmbed(x.shape[-1],max_seq_len=500)(x)
    # Encode
    seqlen = x.shape[1]
    #mask = (jnp.diag(jnp.ones(seqlen - 1), k=(-1)) + jnp.eye(seqlen)).astype(bool) # Attention only on neighbors!
    mask = jnp.ones((seqlen, seqlen), dtype=bool)
    model = Transformer(num_heads=1, num_layers=1, attn_size=x.shape[-1], dropout_rate=0.0)
    embedding = model(x, mask)
    logits = hk.Linear(VOCAB_SIZE)(embedding)
    return logits


In [27]:
params = f.init(key, inputs)
outputs = f.apply(params, key, inputs)

In [28]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

In [29]:

def loss_fn(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    loss = optax.softmax_cross_entropy(logits, labels).mean()
    return loss

def acc(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, inputs, outputs, rng, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params, inputs, outputs, rng)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [30]:
train_seq_len = [10, 50, 100]
for i in range(1000):
    key, subkey, key2 = jrandom.split(jrandom.PRNGKey(i), 3)
    inputs, labels = generate_data(key, 512, train_seq_len[i%len(train_seq_len)], vocab_size=VOCAB_SIZE)
    loss, params, opt_state = update(params, inputs, labels, subkey, opt_state)
    if (i % 100) == 0:
        accuracy = acc(params, inputs, labels, key2)
        print(accuracy, loss)

0.10292969 2.528875
0.3180078 1.9017833
0.5796875 1.1574625
0.5013672 1.2683489
0.63773435 0.87977517
0.70921874 0.70769644
0.7580078 0.7271269
0.7157031 0.6819957
0.7644141 0.55938715
0.8539063 0.48682055


In [31]:
outputs = f.apply(params, key + 2, jax.random.randint(key, (1, 30,1), 0, 10, dtype=jnp.int32))

In [32]:
outputs.argmax(-1)

Array([[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 3, 3, 3, 3, 3, 3,
        4, 4, 4, 4, 4, 5, 5, 5]], dtype=int32)